# Notebook 07 — Results Analysis

**Chapter 4 — Results and Discussion**

This final notebook consolidates all results, generates every Chapter 4 figure and table,
performs the full LSTM vs baseline comparison, and exports the complete results ZIP archive.

## Objectives
- Load trained LSTM + all saved baseline models
- Compare all models on the held-out test set
- Generate the model comparison bar chart
- Display all training curves, confusion matrices, ROC curves
- Compute permutation feature importance
- Produce the full Chapter 4 tables (accuracy, precision, recall, F1, AUC)
- Export all figures and tables as a ZIP archive

---

In [ ]:
import sys, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.utils.helpers import set_global_seed, print_banner
from src.config import get_config
from src.utils.paths import (
    PROCESSED_DATA_DIR, FINAL_MODEL_DIR, BASELINES_DIR,
    FIGURES_DIR, TABLES_DIR, METRICS_DIR
)
from src.utils.serialization import (
    load_processed_arrays, load_keras_model
)
from src.utils.constants import FINAL_MODEL_KERAS, METADATA_JSON

cfg = get_config()
set_global_seed(cfg.seed)
print_banner('Notebook 07 — Results Analysis (Chapter 4)')

## 1. Load Data and Models

In [ ]:
# Load processed test arrays
X_train, X_val, X_test, y_train, y_val, y_test = \
    load_processed_arrays(PROCESSED_DATA_DIR)

# Load metadata
meta_path = FINAL_MODEL_DIR / METADATA_JSON
if not meta_path.exists():
    meta_path = PROCESSED_DATA_DIR / METADATA_JSON
with open(meta_path) as f:
    metadata = json.load(f)

class_names = metadata['class_names']
n_classes   = metadata['n_classes']

print(f'Test set    : {X_test.shape}')
print(f'n_classes   : {n_classes}')
print(f'class_names : {class_names}')

In [ ]:
# Load trained LSTM model
lstm_model = load_keras_model(FINAL_MODEL_DIR / FINAL_MODEL_KERAS)
print(f'LSTM model loaded — {lstm_model.count_params():,} parameters')

In [ ]:
# Load baseline models
from src.models.baseline_models import load_all_baselines

baselines = load_all_baselines(BASELINES_DIR)
print(f'Baseline models loaded: {list(baselines.keys())}')

## 2. Generate All Predictions

In [ ]:
from src.evaluation.metrics import (
    compute_metrics, predict_lstm
)
from src.models.baseline_models import predict_baseline

all_metrics = {}
all_preds   = {}

# LSTM
y_pred_lstm, y_prob_lstm = predict_lstm(lstm_model, X_test)
all_preds['LSTM'] = (y_pred_lstm, y_prob_lstm)
all_metrics['LSTM'] = compute_metrics(
    y_test, y_pred_lstm, y_prob_lstm,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
)

# Baselines
name_map = {
    'random_forest':       'Random Forest',
    'svm':                 'SVM',
    'logistic_regression': 'Logistic Regression',
}
for key, model in baselines.items():
    display_name = name_map.get(key, key)
    y_pred_bl, y_prob_bl = predict_baseline(model, X_test, key)
    all_preds[display_name] = (y_pred_bl, y_prob_bl)
    all_metrics[display_name] = compute_metrics(
        y_test, y_pred_bl, y_prob_bl,
        class_names=class_names,
        dataset='nsl_kdd',
        model_name=display_name,
    )

print(f'Evaluated {len(all_metrics)} models: {list(all_metrics.keys())}')

## 3. Chapter 4 — Performance Comparison Table

In [ ]:
from src.evaluation.comparison import build_comparison_table

comparison_df = build_comparison_table(all_metrics)

# Format for display
numeric_cols = [c for c in comparison_df.columns if c != 'Model']
display_df = comparison_df.copy()
for col in numeric_cols:
    display_df[col] = pd.to_numeric(display_df[col], errors='coerce').round(4)

print('=' * 80)
print('MODEL PERFORMANCE COMPARISON TABLE (Chapter 4)')
print('=' * 80)
print(display_df.to_string(index=False))
print('=' * 80)

## 4. Model Comparison Bar Chart

In [ ]:
from src.evaluation.comparison import plot_model_comparison

path = plot_model_comparison(
    all_metrics,
    metrics_to_plot=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'],
    dataset='nsl_kdd',
)
print(f'Model comparison chart saved: {path}')
display(Image(str(path)))

## 5. LSTM — Classification Report

In [ ]:
from src.evaluation.classification_report import generate_classification_report

report = generate_classification_report(
    y_test, y_pred_lstm,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
)
print(report['report_text'])

## 6. Confusion Matrices

In [ ]:
from src.evaluation.confusion_matrix import (
    plot_confusion_matrix, plot_confusion_matrix_comparison
)

# LSTM confusion matrix (normalised)
path = plot_confusion_matrix(
    y_test, y_pred_lstm,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
    normalize=True,
)
print(f'Confusion matrix saved: {path}')
display(Image(str(path)))

In [ ]:
# Comparison confusion matrices for all models
cm_data = {
    name: {'y_true': y_test, 'y_pred': preds[0]}
    for name, preds in all_preds.items()
}
path = plot_confusion_matrix_comparison(
    cm_data, dataset='nsl_kdd', class_names=class_names
)
print(f'Comparison matrices saved: {path}')
display(Image(str(path)))

## 7. ROC Curves

In [ ]:
from src.evaluation.roc_analysis import (
    compute_roc_curves, plot_roc_curves, save_roc_scores
)

roc_data = compute_roc_curves(y_test, y_prob_lstm, class_names, 'nsl_kdd')

print('AUC Scores per class:')
for cls_name, data in roc_data['per_class'].items():
    print(f'  {cls_name:<14}: {data["auc"]:.4f}')
if 'auc' in roc_data.get('macro', {}):
    print(f'  {"Macro avg":<14}: {roc_data["macro"]["auc"]:.4f}')

path = plot_roc_curves(
    y_test, y_prob_lstm,
    class_names=class_names, dataset='nsl_kdd', model_name='LSTM'
)
save_roc_scores(roc_data)
display(Image(str(path)))

## 8. Precision-Recall Curves

In [ ]:
from src.visualization.plots import plot_precision_recall_curves

path = plot_precision_recall_curves(
    y_test, y_prob_lstm,
    class_names=class_names, dataset='nsl_kdd', model_name='LSTM'
)
display(Image(str(path)))

## 9. Training History (from saved CSV)

In [ ]:
from src.utils.paths import LOGS_DIR

history_path = LOGS_DIR / 'training_history.csv'
if history_path.exists():
    history_df = pd.read_csv(history_path)
    print(f'Training history: {len(history_df)} epochs')
    print(history_df[['epoch', 'loss', 'accuracy', 'val_loss', 'val_accuracy']].tail(5).to_string(index=False))

    from src.visualization.training_curves import plot_combined_training_curves
    hist_dict = history_df.to_dict(orient='list')
    path = plot_combined_training_curves(
        hist_dict, model_name='LSTM', dataset='nsl_kdd'
    )
    display(Image(str(path)))
else:
    print('Training history CSV not found. Run train.py first.')

## 10. Permutation Feature Importance

In [ ]:
from src.data.feature_engineering import (
    compute_permutation_importance, plot_feature_importance
)
from src.utils.serialization import load_feature_names

feature_names = load_feature_names(FINAL_MODEL_DIR / 'feature_names.pkl')

# Use a subset for computational efficiency
n_samples = min(2000, len(X_test))
importance_df = compute_permutation_importance(
    lstm_model,
    X_test[:n_samples], y_test[:n_samples],
    feature_names=feature_names,
    n_repeats=5,
)

print('Top 10 Features by Permutation Importance:')
print(importance_df.head(10)[['feature', 'importance_mean', 'importance_std']].to_string(index=False))

path = plot_feature_importance(importance_df, dataset='nsl_kdd', top_n=15)
display(Image(str(path)))

## 11. Architecture Diagrams

In [ ]:
from src.models.model_utils import plot_lstm_architecture
from src.visualization.architecture_diagrams import (
    plot_system_architecture, plot_data_flow_diagram
)
from src.visualization.plots import plot_preprocessing_pipeline

paths = {
    'LSTM Architecture' : plot_lstm_architecture(
        input_shape=(cfg.sequence.window_size, metadata['n_features']),
        n_classes=n_classes
    ),
    'System Architecture' : plot_system_architecture(),
    'Data Flow'           : plot_data_flow_diagram(),
    'Preprocessing Pipeline': plot_preprocessing_pipeline(),
}

for name, path in paths.items():
    print(f'  {name:<25} → {path}')

display(Image(str(paths['LSTM Architecture'])))

## 12. Save All Evaluation Results

In [ ]:
from src.evaluation.comparison import save_evaluation_results

save_evaluation_results(all_metrics)
print(f'Evaluation results saved to: {METRICS_DIR}')

## 13. Export Chapter 4 ZIP Archive

In [ ]:
from src.visualization.dashboard import export_chapter4_zip

zip_path = export_chapter4_zip()
print(f'Chapter 4 ZIP exported: {zip_path}')

## 14. Final Summary

In [ ]:
lstm_m = all_metrics['LSTM']

print('=' * 60)
print('CHAPTER 4 — FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Dataset           : NSL-KDD')
print(f'Test samples      : {len(y_test):,}')
print(f'Window size       : {X_test.shape[1]}')
print(f'Features          : {X_test.shape[2]}')
print()
print('LSTM Performance:')
print(f'  Accuracy          : {lstm_m["accuracy"]:.4f}')
print(f'  Precision (macro) : {lstm_m["precision_macro"]:.4f}')
print(f'  Recall    (macro) : {lstm_m["recall_macro"]:.4f}')
print(f'  F1-Score  (macro) : {lstm_m["f1_macro"]:.4f}')
print(f'  F1-Score  (wtd)   : {lstm_m["f1_weighted"]:.4f}')
if lstm_m.get('roc_auc'):
    print(f'  ROC-AUC           : {lstm_m["roc_auc"]:.4f}')
print()
print('All figures and tables saved to:')
print(f'  Figures  : reports/figures/')
print(f'  Tables   : reports/tables/')
print(f'  Metrics  : reports/metrics/')
print(f'  ZIP      : {zip_path}')
print('=' * 60)